# 06_gradient_based_planning.ipynb – GRASP-style Gradient Planning

**Goal**  
Compare random shooting vs gradient-based optimization of an action sequence in a learned latent world model.

This notebook loads the model checkpoint from Notebook 05 (`checkpoints/value_guided_cartpole.pt`) and performs planning with a differentiable action-parameterization (soft action probabilities).

**Reference**
- Parallel Stochastic Gradient-Based Planning (GRASP): https://arxiv.org/abs/2602.00475

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial import ValueHead
from viz.planning_rollout_viz import plot_episode_rewards

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class StateEncoder(nn.Module):
    def __init__(self, in_dim=4, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.net(x)

class LatentDynamics(nn.Module):
    def __init__(self, latent_dim=64, n_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + n_actions, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.n_actions = n_actions

    def forward(self, z, a):
        if a.dtype in (torch.int32, torch.int64, torch.int16):
            a_feat = F.one_hot(a.long(), num_classes=self.n_actions).float()
        else:
            a_feat = a.float()
        return self.net(torch.cat([z, a_feat], dim=-1))

ckpt_path = Path("notebooks/checkpoints/value_guided_cartpole.pt")
if not ckpt_path.exists():
    raise FileNotFoundError("Missing checkpoints/value_guided_cartpole.pt. Run notebook 05 first.")

ckpt = torch.load(ckpt_path, map_location=device)
latent_dim = int(ckpt.get("latent_dim", 64))
encoder = StateEncoder(latent_dim=latent_dim).to(device)
dynamics = LatentDynamics(latent_dim=latent_dim, n_actions=2).to(device)
value_head = ValueHead(latent_dim=latent_dim, hidden_dim=128).to(device)

encoder.load_state_dict(ckpt["encoder"])
dynamics.load_state_dict(ckpt["dynamics"])
value_head.load_state_dict(ckpt["value_head"])

encoder.eval(); dynamics.eval(); value_head.eval()
print("Loaded value-guided model checkpoint.")

In [ ]:
@torch.no_grad()
def score_action_sequences(z0, action_sequences):
    actions = torch.as_tensor(action_sequences, dtype=torch.long, device=device)
    z = z0.expand(actions.shape[0], -1)
    scores = torch.zeros(actions.shape[0], dtype=z.dtype, device=device)
    discount = 1.0
    for step in range(actions.shape[1]):
        z = dynamics(z, actions[:, step])
        scores += discount * value_head(z).squeeze(-1)
        discount *= 0.98
    return scores

@torch.no_grad()
def random_shooting_action(obs_np, horizon=14, n_candidates=256):
    z0 = encoder(torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0))
    candidates = np.random.randint(0, 2, size=(n_candidates, horizon))
    scores = score_action_sequences(z0, candidates)
    best = int(scores.argmax().item())
    return int(candidates[best, 0])

def gradient_plan_action(obs_np, horizon=14, iters=25, lr=0.15):
    z0 = encoder(torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0)).detach()
    logits = torch.zeros(horizon, 2, device=device, requires_grad=True)
    optimizer = optim.Adam([logits], lr=lr)

    for _ in range(iters):
        probs = logits.softmax(dim=-1)
        z = z0
        total = 0.0
        disc = 1.0
        for t in range(horizon):
            z = dynamics(z, probs[t].unsqueeze(0))
            total = total + disc * value_head(z).mean()
            disc *= 0.98
        loss = -total
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    best_action = int(logits.softmax(dim=-1)[0].argmax().item())
    return best_action

In [ ]:
# Compare both planners on the same reset seeds and repeatable random candidates.
def run_policy(env, planner_fn, seeds):
    rewards = []
    for episode_seed in seeds:
        episode_seed = int(episode_seed)
        env.action_space.seed(episode_seed)
        np.random.seed(episode_seed)
        obs, _ = env.reset(seed=episode_seed)
        done = False
        ep_reward = 0.0
        while not done:
            action = planner_fn(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_reward += reward
        rewards.append(ep_reward)
    return np.array(rewards, dtype=np.float32)

eval_seeds = list(range(100, 120))
env = gym.make("CartPole-v1")
rand_rewards = run_policy(env, planner_fn=lambda obs: random_shooting_action(obs), seeds=eval_seeds)
grad_rewards = run_policy(env, planner_fn=lambda obs: gradient_plan_action(obs), seeds=eval_seeds)
env.close()

print("evaluation reset/candidate seeds:", eval_seeds)
print("Random-shooting returns:", rand_rewards.astype(int).tolist())
print("Gradient-planning returns:", grad_rewards.astype(int).tolist())
print(f"Random-shooting mean reward: {rand_rewards.mean():.2f}")
print(f"Gradient-planning mean reward: {grad_rewards.mean():.2f}")

plot_episode_rewards(rand_rewards, title="Random-shooting rewards")
plt.show()
plot_episode_rewards(grad_rewards, title="Gradient-planning rewards")
plt.show()

## Suggested exercises

1. Tune gradient-planning optimization steps (`iters`) and horizon to study quality-vs-compute.
2. Replace softmax action relaxation with Gumbel-Softmax.
3. Compare first-action-only replanning vs full sequence execution.